<a href="https://colab.research.google.com/github/aranavkumar/Veloura/blob/main/06_Veloura_Reasoning_Context_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# VELOURA - PHASE 6
# REASONING & CONTEXT AGENT
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime

# Project directory
PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

# Input directory
INPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "agent_logs"
)

# Output directories
OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs",
    "reasoning_context"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "reasoning_context"
)

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Veloura Reasoning & Context Agent")
print("=" * 55)

print("Project directory:", PROJECT_DIR)
print("Input directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Report directory:", REPORT_DIR)

print("\nSUCCESS: Phase 6 environment initialized!")


Veloura Reasoning & Context Agent
Project directory: /content/drive/MyDrive/Veloura_Project
Input directory: /content/drive/MyDrive/Veloura_Project/outputs/agent_logs
Output directory: /content/drive/MyDrive/Veloura_Project/outputs/reasoning_context
Report directory: /content/drive/MyDrive/Veloura_Project/reports/reasoning_context

SUCCESS: Phase 6 environment initialized!


In [ ]:

# PHASE 6 OUTPUT CHECK

import os

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

OUTPUT_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "reasoning_context",
    "reasoning_context_sample_logs.json"
)

print("Phase 6 output exists:", os.path.exists(OUTPUT_JSON))

if os.path.exists(OUTPUT_JSON):
    print("File size:", os.path.getsize(OUTPUT_JSON), "bytes")
    print("Output path:", OUTPUT_JSON)


Phase 6 output exists: False


In [ ]:

# ============================================================
# PHASE 6: OUTPUT RECOVERY FOR PHASE 7
# ============================================================

import os
import json
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "outputs", "reasoning_context"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR, "reports", "reasoning_context"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Check original in-memory records
if "context_records" not in globals():
    raise RuntimeError(
        "context_records unavailable. "
        "Rerun Phase 6 Cells 2-5 before exporting."
    )

assert len(context_records) == 6

# Verify required information before export
for record in context_records:
    assert "downstream_evidence" in record
    assert record["clinical_constraints"][
        "automatic_acceptance"
    ] is False

# Export JSON
JSON_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "reasoning_context_sample_logs.json"
)

with open(JSON_OUTPUT, "w") as f:
    json.dump(context_records, f, indent=4, default=str)

# Rebuild CSV from original context records
summary = pd.DataFrame([
    {
        "image_id": r["image_id"],
        "age": r["patient_context"]["age"],
        "sex": r["patient_context"]["sex"],
        "lesion_location": r["lesion_context"]["location"],
        "missing_information": ", ".join(
            r["missing_information"]
        ),
        "completeness_score": r["context_completeness"]["score"],
        "completeness_status": r["context_completeness"]["status"],
        "analysis_status": r["lesion_context"]["analysis_status"],
        "feature_status": r["lesion_context"]["feature_status"],
        "review_required": r["clinical_constraints"]["requires_review"],
        "automatic_acceptance": r["clinical_constraints"]["automatic_acceptance"]
    }
    for r in context_records
])

CSV_OUTPUT = os.path.join(
    REPORT_DIR,
    "reasoning_context_sample_summary.csv"
)

summary.to_csv(CSV_OUTPUT, index=False)

# Verify JSON export
with open(JSON_OUTPUT, "r") as f:
    verification_data = json.load(f)

assert len(verification_data) == 6
assert all(
    r["clinical_constraints"]["requires_review"] is True
    for r in verification_data
)

print("=" * 60)
print("PHASE 6 RECOVERY REPORT")
print("=" * 60)

print("JSON exists:", os.path.exists(JSON_OUTPUT))
print("CSV exists:", os.path.exists(CSV_OUTPUT))
print("Records restored:", len(verification_data))
print("Review requirements preserved: PASS")
print("Automatic acceptance disabled: PASS")

print("\nSUCCESS: PHASE 6 OUTPUT RESTORED!")


RuntimeError: context_records unavailable. Rerun Phase 6 Cells 2-5 before exporting.

In [ ]:

# ============================================================
# PHASE 6 RECOVERY: REBUILD FROM ORIGINAL PHASE 5 LOGS
# ============================================================

import os
import json
import pandas as pd
from datetime import datetime, timezone

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

PHASE5_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "agent_logs",
    "lesion_analysis_sample_logs.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "outputs", "reasoning_context"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR, "reports", "reasoning_context"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Load original Phase 5 data
if not os.path.exists(PHASE5_JSON):
    raise FileNotFoundError(PHASE5_JSON)

with open(PHASE5_JSON, "r") as f:
    phase5_data = json.load(f)

assert len(phase5_data) == 6

context_records = []

for source in phase5_data:

    context = {
        "agent": "Veloura Reasoning & Context Agent",
        "image_id": source["image_id"],

        "patient_context": {
            "age": None,
            "sex": None
        },

        "lesion_context": {
            "location": None,
            "analysis_status": source["status"],
            "segmentation_method": source["segmentation_method"],
            "feature_status": source["features"].get(
                "feature_status",
                "EXPERIMENTAL_UNVALIDATED"
            )
        },

        "source_agent": source["agent"],

        "source_evidence": {
            "evidence_summary": source["evidence_summary"],
            "reliability_flags": source["reliability_flags"],
            "warnings": source["warnings"]
        },

        "clinical_constraints": {
            "independent_diagnosis": False,
            "automatic_acceptance": False,
            "requires_review": True
        }
    }

    # Missing information
    context["missing_information"] = [
        "age", "sex", "lesion_location"
    ]

    context["context_completeness"] = {
        "total_fields": 3,
        "available_fields": 0,
        "missing_fields": 3,
        "score": 0.0,
        "status": "INCOMPLETE",
        "score_interpretation": "DATA_AVAILABILITY_ONLY"
    }

    # Safety evidence
    safety_evidence = {
        "image_id": source["image_id"],
        "review_required": True,
        "automatic_acceptance": False,
        "missing_information": context["missing_information"],
        "context_completeness": context["context_completeness"],
        "reliability_flags": source["reliability_flags"],
        "warnings": source["warnings"],
        "feature_status": context["lesion_context"]["feature_status"],
        "clinical_constraints": context["clinical_constraints"]
    }

    # Coordinator evidence
    coordinator_evidence = {
        "image_id": source["image_id"],
        "patient_context": context["patient_context"],
        "lesion_context": context["lesion_context"],
        "missing_information": context["missing_information"],
        "context_completeness": context["context_completeness"],
        "source_agent": source["agent"],
        "next_action": "MANUAL_REVIEW_REQUIRED"
    }

    context["downstream_evidence"] = {
        "safety_agent": safety_evidence,
        "coordinator": coordinator_evidence
    }

    context_records.append(context)

# Export recovered JSON
JSON_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "reasoning_context_sample_logs.json"
)

with open(JSON_OUTPUT, "w") as f:
    json.dump(context_records, f, indent=4, default=str)

# Export recovered CSV
summary = pd.DataFrame([
    {
        "image_id": r["image_id"],
        "age": None,
        "sex": None,
        "lesion_location": None,
        "missing_information": "age, sex, lesion_location",
        "completeness_score": 0.0,
        "completeness_status": "INCOMPLETE",
        "analysis_status": r["lesion_context"]["analysis_status"],
        "feature_status": r["lesion_context"]["feature_status"],
        "review_required": True,
        "automatic_acceptance": False
    }
    for r in context_records
])

CSV_OUTPUT = os.path.join(
    REPORT_DIR,
    "reasoning_context_sample_summary.csv"
)

summary.to_csv(CSV_OUTPUT, index=False)

# Verify
with open(JSON_OUTPUT, "r") as f:
    verified = json.load(f)

assert len(verified) == 6
assert all(
    r["clinical_constraints"]["requires_review"] is True
    for r in verified
)
assert all(
    r["clinical_constraints"]["automatic_acceptance"] is False
    for r in verified
)

print("=" * 60)
print("PHASE 6 RECOVERY COMPLETED")
print("=" * 60)

print("Records recovered:", len(verified))
print("JSON saved:", os.path.exists(JSON_OUTPUT))
print("CSV saved:", os.path.exists(CSV_OUTPUT))
print("Missing information preserved: PASS")
print("Manual review preserved: PASS")
print("Automatic acceptance disabled: PASS")

print("\nSUCCESS: PHASE 6 OUTPUTS RESTORED!")


FileNotFoundError: /content/drive/MyDrive/Veloura_Project/outputs/agent_logs/lesion_analysis_sample_logs.json

In [ ]:

# ============================================================
# VELOURA: GLOBAL DRIVE DIAGNOSTICS
# ============================================================

import os

from google.colab import drive

# Mount at a fresh location to avoid existing mount conflicts
MOUNT_DIR = "/content/veloura_drive"

os.makedirs(MOUNT_DIR, exist_ok=True)

if not os.path.exists(os.path.join(MOUNT_DIR, "MyDrive")):
    drive.mount(MOUNT_DIR)

SEARCH_ROOT = os.path.join(MOUNT_DIR, "MyDrive")

print("=" * 60)
print("VELOURA GLOBAL DRIVE INSPECTION")
print("=" * 60)

print("Drive accessible:", os.path.exists(SEARCH_ROOT))

if os.path.exists(SEARCH_ROOT):

    for root, dirs, files in os.walk(SEARCH_ROOT):

        dirs[:] = [
            d for d in dirs
            if d not in [".cache", "node_modules"]
        ]

        depth = root.replace(SEARCH_ROOT, "").count(os.sep)

        if depth > 5:
            dirs[:] = []
            continue

        print("\n" + "  " * depth + os.path.basename(root) + "/")

        for filename in files:
            if (
                "lesion_analysis" in filename
                or "reasoning_context" in filename
            ):
                print("  " * (depth + 1) + filename)

print("\nDRIVE INSPECTION COMPLETED!")


Mounted at /content/veloura_drive
VELOURA GLOBAL DRIVE INSPECTION
Drive accessible: True

MyDrive/

  Colab Notebooks/

  Veloura_Project/

    datasets/

      training/

        extracted/

          Health/

      evaluation/

        extracted/

          Health/

    models/

      checkpoints/

      final_models/

    notebooks/

    reports/

      dataset_analysis/

      model_evaluation/

      image_quality/

      lesion_analysis/
        lesion_analysis_sample_summary.csv

      reasoning_context/
        reasoning_context_sample_summary.csv

    outputs/

      predictions/

      agent_logs/
        lesion_analysis_sample_logs.json

      reasoning_context/
        reasoning_context_sample_logs.json

    documentation/

DRIVE INSPECTION COMPLETED!


In [ ]:
print(os.path.exists(
    "/content/drive/MyDrive/Veloura_Project/outputs/agent_logs/lesion_analysis_sample_logs.json"
))

False


In [ ]:
import os

print("Phase 6 Drive check")

print("Drive mounted:",
      os.path.exists("/content/drive/MyDrive"))

print("Project contents:",
      os.listdir(
          "/content/drive/MyDrive/Veloura_Project"
      ))

print("Agent logs directory:",
      os.path.exists(
          "/content/drive/MyDrive/Veloura_Project/outputs/agent_logs"
      ))

Phase 6 Drive check
Drive mounted: True
Project contents: ['datasets', 'models', 'notebooks', 'reports', 'outputs', 'documentation', 'README.md', 'requirements.txt']
Agent logs directory: True


In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [ ]:

# ============================================================
# CELL 2: PHASE 5 INPUT LOADER & SCHEMA VERIFICATION
# ============================================================

import os
import json

INPUT_JSON = os.path.join(
    INPUT_DIR,
    "lesion_analysis_sample_logs.json"
)

# Verify input file
if not os.path.exists(INPUT_JSON):
    raise FileNotFoundError(
        f"Phase 5 JSON file not found: {INPUT_JSON}"
    )

# Load original Phase 5 records
with open(INPUT_JSON, "r") as file:
    phase5_data = json.load(file)

print("=" * 60)
print("VELOURA PHASE 6 - INPUT VERIFICATION")
print("=" * 60)

# Verify structure
if not isinstance(phase5_data, list):
    raise TypeError(
        "Expected Phase 5 JSON to contain a list of records."
    )

print("JSON data type:", type(phase5_data).__name__)
print("Total records:", len(phase5_data))

# Required fields
required_fields = [
    "agent",
    "image_id",
    "status",
    "segmentation_method",
    "segmentation",
    "features",
    "evidence_summary",
    "reliability_flags",
    "warnings",
    "clinical_interpretation",
    "automatic_acceptance"
]

# Validate records
for index, record in enumerate(phase5_data):

    missing_fields = [
        field for field in required_fields
        if field not in record
    ]

    if missing_fields:
        raise ValueError(
            f"Record {index} missing fields: {missing_fields}"
        )

# Display first record structure
print("\nFIRST RECORD KEYS:")
print(list(phase5_data[0].keys()))

print("\nIMAGE IDS:")
for record in phase5_data:
    print("-", record["image_id"])

# Verify safety flags
print("\nSTATUS DISTRIBUTION:")

for status in sorted(set(
    record["status"] for record in phase5_data
)):
    print(
        status,
        sum(r["status"] == status for r in phase5_data)
    )

print("\nAUTOMATIC ACCEPTANCE VALUES:")
print(set(
    record["automatic_acceptance"]
    for record in phase5_data
))

print("\nSUCCESS: PHASE 5 INPUT SCHEMA VERIFIED!")


FileNotFoundError: Phase 5 JSON file not found: /content/drive/MyDrive/Veloura_Project/outputs/agent_logs/lesion_analysis_sample_logs.json

In [ ]:

# ============================================================
# CELL 2A: PHASE 6 DRIVE DIAGNOSTICS
# ============================================================

import os

print("DRIVE STATUS")
print("=" * 50)

print("Drive directory exists:",
      os.path.exists("/content/drive"))

print("MyDrive exists:",
      os.path.exists("/content/drive/MyDrive"))

print("Project exists:",
      os.path.exists(
          "/content/drive/MyDrive/Veloura_Project"
      ))

if os.path.exists("/content/drive/MyDrive"):
    print("\nMyDrive contents:")
    print(os.listdir("/content/drive/MyDrive")[:20])


DRIVE STATUS
Drive directory exists: True
MyDrive exists: True
Project exists: True

MyDrive contents:
['Veloura_Project']


In [ ]:

# ============================================================
# CELL 2B: CHECK PHASE 5 OUTPUT PATHS
# ============================================================

import os

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

paths_to_check = [
    "outputs",
    "outputs/agent_logs",
    "outputs/agent_logs/lesion_analysis_sample_logs.json",
    "reports",
    "reports/lesion_analysis",
    "reports/lesion_analysis/lesion_analysis_sample_summary.csv"
]

for relative_path in paths_to_check:

    full_path = os.path.join(
        PROJECT_DIR,
        relative_path
    )

    print(
        f"{relative_path}:",
        "EXISTS" if os.path.exists(full_path)
        else "NOT FOUND"
    )

    if os.path.isdir(full_path):
        print("  Contents:", os.listdir(full_path))

print("\nPATH DIAGNOSTICS COMPLETED!")


outputs: EXISTS
  Contents: ['reasoning_context']
outputs/agent_logs: NOT FOUND
outputs/agent_logs/lesion_analysis_sample_logs.json: NOT FOUND
reports: EXISTS
  Contents: ['reasoning_context']
reports/lesion_analysis: NOT FOUND
reports/lesion_analysis/lesion_analysis_sample_summary.csv: NOT FOUND

PATH DIAGNOSTICS COMPLETED!


In [ ]:

# ============================================================
# CELL 2: PHASE 5 INPUT LOADER & SCHEMA VERIFICATION
# ============================================================

INPUT_JSON = os.path.join(
    INPUT_DIR,
    "lesion_analysis_sample_logs.json"
)

# Check input file
if not os.path.exists(INPUT_JSON):
    raise FileNotFoundError(
        f"Phase 5 JSON file not found: {INPUT_JSON}"
    )

# Load JSON records
with open(INPUT_JSON, "r") as file:
    phase5_data = json.load(file)

print("Phase 5 JSON loaded successfully!")

# Inspect data structure
print("\nJSON DATA TYPE:", type(phase5_data))

if isinstance(phase5_data, list):
    print("Total records:", len(phase5_data))
    sample_record = phase5_data[0]

elif isinstance(phase5_data, dict):
    print("Top-level keys:", list(phase5_data.keys()))

    # Identify record collection
    sample_record = phase5_data

else:
    raise ValueError("Unexpected Phase 5 JSON structure")

# Display sample record
print("\nSAMPLE RECORD:")
print(json.dumps(sample_record, indent=4))

print("\nSUCCESS: Phase 5 input inspection completed!")


FileNotFoundError: Phase 5 JSON file not found: /content/drive/MyDrive/Veloura_Project/outputs/agent_logs/lesion_analysis_sample_logs.json

In [ ]:

# ============================================================
# CELL 2A: GOOGLE DRIVE & PHASE 5 FILE DIAGNOSTICS
# ============================================================

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

print("\nPROJECT DIRECTORY EXISTS:",
      os.path.exists(PROJECT_DIR))

if os.path.exists(PROJECT_DIR):

    print("\nPROJECT DIRECTORY CONTENTS:")

    for item in os.listdir(PROJECT_DIR):
        print("-", item)

else:
    print("\nVeloura_Project directory not found!")


ValueError: Mountpoint must not already contain files

In [ ]:

import os

print("Drive path exists:",
      os.path.exists("/content/drive"))

print("MyDrive exists:",
      os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive"):
    print("\nDrive directory contents:")
    print(os.listdir("/content/drive")[:20])



Drive path exists: True
MyDrive exists: True

Drive directory contents:
['MyDrive']


In [ ]:

PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

print("Project exists:", os.path.exists(PROJECT_DIR))

if os.path.exists(PROJECT_DIR):
    print("\nProject folders:")
    print(os.listdir(PROJECT_DIR))
else:
    print("\nVeloura_Project not found.")


Project exists: True

Project folders:
['outputs', 'reports']


In [ ]:

# ============================================================
# CELL 2C: INSPECT VELOURA PROJECT FILES
# ============================================================

import os

for folder in ["outputs", "reports"]:

    folder_path = os.path.join(PROJECT_DIR, folder)

    print("\n" + "=" * 55)
    print(f"{folder.upper()} DIRECTORY")
    print("=" * 55)

    for root, dirs, files in os.walk(folder_path):

        level = root.replace(folder_path, "").count(os.sep)

        indent = "    " * level

        print(f"{indent}{os.path.basename(root)}/")

        for filename in files:
            print(f"{indent}    {filename}")

print("\nSUCCESS: Project inspection completed!")



OUTPUTS DIRECTORY
outputs/
    reasoning_context/

REPORTS DIRECTORY
reports/
    reasoning_context/

SUCCESS: Project inspection completed!


In [ ]:

# ============================================================
# CELL 2D: GLOBAL PHASE 5 OUTPUT SEARCH
# ============================================================

import os

search_root = "/content/drive/MyDrive"

target_files = [
    "lesion_analysis_sample_logs.json",
    "lesion_analysis_sample_summary.csv"
]

found_files = {name: [] for name in target_files}

for root, dirs, files in os.walk(search_root):

    dirs[:] = [
        d for d in dirs
        if d not in ["node_modules", ".cache"]
    ]

    for filename in target_files:

        if filename in files:
            found_files[filename].append(
                os.path.join(root, filename)
            )

print("=" * 60)
print("VELOURA PHASE 5 GLOBAL FILE SEARCH")
print("=" * 60)

for filename, paths in found_files.items():

    print(f"\n{filename}")

    if paths:
        for path in paths:
            print("FOUND:", path)
    else:
        print("NOT FOUND")

print("\nSEARCH COMPLETED!")


VELOURA PHASE 5 GLOBAL FILE SEARCH

lesion_analysis_sample_logs.json
NOT FOUND

lesion_analysis_sample_summary.csv
NOT FOUND

SEARCH COMPLETED!


In [ ]:

# ============================================================
# CELL 2: PHASE 5 INPUT LOADER & SCHEMA VERIFICATION
# ============================================================

import os
import json

# Explicit project configuration
PROJECT_DIR = "/content/drive/MyDrive/Veloura_Project"

INPUT_JSON = os.path.join(
    PROJECT_DIR,
    "outputs",
    "agent_logs",
    "lesion_analysis_sample_logs.json"
)

print("Input path:", INPUT_JSON)
print("File exists:", os.path.exists(INPUT_JSON))

if not os.path.exists(INPUT_JSON):
    raise FileNotFoundError(
        f"Phase 5 JSON file not found: {INPUT_JSON}"
    )

# Load Phase 5 records
with open(INPUT_JSON, "r") as file:
    phase5_data = json.load(file)

# Verify structure
if not isinstance(phase5_data, list):
    raise TypeError("Expected a list of JSON records.")

if len(phase5_data) == 0:
    raise ValueError("Phase 5 JSON contains no records.")

print("\nJSON DATA TYPE:", type(phase5_data).__name__)
print("TOTAL RECORDS:", len(phase5_data))

# Required fields
required_fields = [
    "agent",
    "image_id",
    "status",
    "segmentation_method",
    "segmentation",
    "features",
    "evidence_summary",
    "reliability_flags",
    "warnings",
    "clinical_interpretation",
    "automatic_acceptance"
]

# Schema validation
for index, record in enumerate(phase5_data):

    if not isinstance(record, dict):
        raise TypeError(
            f"Record {index} is not a dictionary."
        )

    missing = [
        field for field in required_fields
        if field not in record
    ]

    if missing:
        raise ValueError(
            f"Record {index} missing fields: {missing}"
        )

print("\nIMAGE IDS:")

for record in phase5_data:
    print("-", record["image_id"])

print("\nSTATUS DISTRIBUTION:")

for status in sorted({
    record["status"] for record in phase5_data
}):
    count = sum(
        record["status"] == status
        for record in phase5_data
    )
    print(status, ":", count)

print("\nAUTOMATIC ACCEPTANCE:")
print([
    record["automatic_acceptance"]
    for record in phase5_data
])

print("\nSUCCESS: PHASE 5 INPUT SCHEMA VERIFIED!")


Input path: /content/drive/MyDrive/Veloura_Project/outputs/agent_logs/lesion_analysis_sample_logs.json
File exists: True

JSON DATA TYPE: list
TOTAL RECORDS: 6

IMAGE IDS:
- ISIC_0033038
- ISIC_0025481
- ISIC_0034119
- ISIC_0031437
- ISIC_0031157
- ISIC_0028145

STATUS DISTRIBUTION:
REVIEW_REQUIRED : 6

AUTOMATIC ACCEPTANCE:
[False, False, False, False, False, False]

SUCCESS: PHASE 5 INPUT SCHEMA VERIFIED!


In [ ]:

# ============================================================
# CELL 3: PATIENT & LESION CONTEXT SCHEMA
# ============================================================

from datetime import datetime, timezone

CONTEXT_AGENT_NAME = "Veloura Reasoning & Context Agent"

def create_context_schema(phase5_record, patient_data=None):

    patient_data = patient_data or {}

    # Preserve missing information
    age = patient_data.get("age")
    sex = patient_data.get("sex")
    location = patient_data.get("lesion_location")

    # Validate age
    if age is not None:
        if isinstance(age, bool) or not isinstance(age, int):
            raise ValueError("Age must be an integer or None.")

        if not 0 <= age <= 120:
            raise ValueError("Age must be between 0 and 120.")

    # Validate sex without inferring it
    allowed_sex_values = {
        "male", "female", "intersex",
        "other", "prefer_not_to_say"
    }

    if sex is not None:
        if not isinstance(sex, str):
            raise ValueError("Sex must be a string or None.")

        sex = sex.strip().lower()

        if sex not in allowed_sex_values:
            raise ValueError("Invalid sex value.")

    # Validate lesion location
    if location is not None:
        if not isinstance(location, str) or not location.strip():
            raise ValueError(
                "Lesion location must be a non-empty string or None."
            )

        location = location.strip()

    return {
        "agent": CONTEXT_AGENT_NAME,
        "image_id": phase5_record["image_id"],

        "patient_context": {
            "age": age,
            "sex": sex
        },

        "lesion_context": {
            "location": location,
            "analysis_status": phase5_record["status"],
            "segmentation_method": phase5_record["segmentation_method"],
            "feature_status": phase5_record["features"].get(
                "feature_status",
                "NOT_CONFIRMED"
            )
        },

        "source_agent": phase5_record["agent"],

        "source_evidence": {
            "evidence_summary": phase5_record["evidence_summary"],
            "reliability_flags": phase5_record["reliability_flags"],
            "warnings": phase5_record["warnings"]
        },

        "clinical_constraints": {
            "independent_diagnosis": False,
            "automatic_acceptance": False,
            "requires_review": True
        },

        "generated_at": datetime.now(
            timezone.utc
        ).isoformat()
    }


# Process six records without inventing patient information
context_records = [
    create_context_schema(record)
    for record in phase5_data
]

print("Context records generated:", len(context_records))
print("Sample context record:")
print(json.dumps(context_records[0], indent=4, default=str))

print("\nSUCCESS: PATIENT & LESION CONTEXT SCHEMA CREATED!")


Context records generated: 6
Sample context record:
{
    "agent": "Veloura Reasoning & Context Agent",
    "image_id": "ISIC_0033038",
    "patient_context": {
        "age": null,
        "sex": null
    },
    "lesion_context": {
        "location": null,
        "analysis_status": "REVIEW_REQUIRED",
        "segmentation_method": "GRABCUT_V1",
        "feature_status": "EXPERIMENTAL_UNVALIDATED"
    },
    "source_agent": "Veloura Lesion Analysis Agent",
    "source_evidence": {
        "evidence_summary": {
            "asymmetry_estimate": 0.1152,
            "boundary_irregularity_estimate": 0.4536,
            "circularity_estimate": 0.5464,
            "colour_variation": {
                "hue_circular_dispersion": 0.0401,
                "saturation_std": 35.4,
                "brightness_std": 18.22
            }
        },
        "reliability_flags": [],
        "warnings": [
            "SEGMENTATION_NOT_VALIDATED_AGAINST_EXPERT_MASKS",
            "VISUAL_FEATURES_ARE_E

In [ ]:

# ============================================================
# CELL 4: MISSING INFORMATION & CONTEXT COMPLETENESS
# ============================================================

import pandas as pd

def evaluate_context_completeness(context):

    patient = context["patient_context"]
    lesion = context["lesion_context"]

    required_fields = {
        "age": patient.get("age"),
        "sex": patient.get("sex"),
        "lesion_location": lesion.get("location")
    }

    missing_information = [
        field
        for field, value in required_fields.items()
        if value is None
    ]

    available_count = (
        len(required_fields) - len(missing_information)
    )

    completeness_score = round(
        available_count / len(required_fields) * 100,
        2
    )

    if completeness_score == 100:
        completeness_status = "COMPLETE"
    elif completeness_score > 0:
        completeness_status = "PARTIALLY_COMPLETE"
    else:
        completeness_status = "INCOMPLETE"

    context["missing_information"] = missing_information

    context["context_completeness"] = {
        "total_fields": len(required_fields),
        "available_fields": available_count,
        "missing_fields": len(missing_information),
        "score": completeness_score,
        "status": completeness_status,
        "score_interpretation": "DATA_AVAILABILITY_ONLY"
    }

    return context


# Evaluate records
context_records = [
    evaluate_context_completeness(record)
    for record in context_records
]

# Create summary
completeness_summary = pd.DataFrame([
    {
        "image_id": record["image_id"],
        "missing_information": ", ".join(
            record["missing_information"]
        ),
        "completeness_score": record[
            "context_completeness"
        ]["score"],
        "completeness_status": record[
            "context_completeness"
        ]["status"]
    }
    for record in context_records
])

print("=" * 65)
print("VELOURA CONTEXT COMPLETENESS REPORT")
print("=" * 65)

print(completeness_summary.to_string(index=False))

print("\nSTATUS DISTRIBUTION:")
print(completeness_summary[
    "completeness_status"
].value_counts().to_string())

print("\nAVERAGE CONTEXT COMPLETENESS:",
      completeness_summary["completeness_score"].mean(), "%")

print("\nSUCCESS: CONTEXT COMPLETENESS EVALUATION COMPLETED!")


VELOURA CONTEXT COMPLETENESS REPORT
    image_id       missing_information  completeness_score completeness_status
ISIC_0033038 age, sex, lesion_location                 0.0          INCOMPLETE
ISIC_0025481 age, sex, lesion_location                 0.0          INCOMPLETE
ISIC_0034119 age, sex, lesion_location                 0.0          INCOMPLETE
ISIC_0031437 age, sex, lesion_location                 0.0          INCOMPLETE
ISIC_0031157 age, sex, lesion_location                 0.0          INCOMPLETE
ISIC_0028145 age, sex, lesion_location                 0.0          INCOMPLETE

STATUS DISTRIBUTION:
completeness_status
INCOMPLETE    6

AVERAGE CONTEXT COMPLETENESS: 0.0 %

SUCCESS: CONTEXT COMPLETENESS EVALUATION COMPLETED!


In [ ]:

# ============================================================
# CELL 5: STRUCTURED EVIDENCE BUILDER
# ============================================================

def build_structured_evidence(context):

    safety_evidence = {
        "image_id": context["image_id"],
        "review_required": True,
        "automatic_acceptance": False,

        "missing_information": context[
            "missing_information"
        ],

        "context_completeness": context[
            "context_completeness"
        ],

        "reliability_flags": context[
            "source_evidence"
        ]["reliability_flags"],

        "warnings": context[
            "source_evidence"
        ]["warnings"],

        "feature_status": context[
            "lesion_context"
        ]["feature_status"],

        "clinical_constraints": context[
            "clinical_constraints"
        ]
    }

    coordinator_evidence = {
        "image_id": context["image_id"],

        "patient_context": context[
            "patient_context"
        ],

        "lesion_context": context[
            "lesion_context"
        ],

        "missing_information": context[
            "missing_information"
        ],

        "context_completeness": context[
            "context_completeness"
        ],

        "source_agent": context["source_agent"],

        "next_action": "MANUAL_REVIEW_REQUIRED"
    }

    context["downstream_evidence"] = {
        "safety_agent": safety_evidence,
        "coordinator": coordinator_evidence
    }

    return context


# Build evidence for all records
context_records = [
    build_structured_evidence(record)
    for record in context_records
]

# Verify evidence generation
assert len(context_records) == 6

for record in context_records:

    assert "safety_agent" in record["downstream_evidence"]
    assert "coordinator" in record["downstream_evidence"]

    assert record["downstream_evidence"][
        "safety_agent"
    ]["review_required"] is True

    assert record["downstream_evidence"][
        "safety_agent"
    ]["automatic_acceptance"] is False

print("=" * 65)
print("VELOURA STRUCTURED EVIDENCE REPORT")
print("=" * 65)

print("Records processed:", len(context_records))
print("Safety evidence generated:", len(context_records))
print("Coordinator evidence generated:", len(context_records))

print("\nSAMPLE SAFETY AGENT EVIDENCE:")
print(json.dumps(
    context_records[0]["downstream_evidence"]["safety_agent"],
    indent=4,
    default=str
))

print("\nSAMPLE COORDINATOR EVIDENCE:")
print(json.dumps(
    context_records[0]["downstream_evidence"]["coordinator"],
    indent=4,
    default=str
))

print("\nSUCCESS: STRUCTURED EVIDENCE GENERATED!")


VELOURA STRUCTURED EVIDENCE REPORT
Records processed: 6
Safety evidence generated: 6
Coordinator evidence generated: 6

SAMPLE SAFETY AGENT EVIDENCE:
{
    "image_id": "ISIC_0033038",
    "review_required": true,
    "automatic_acceptance": false,
    "missing_information": [
        "age",
        "sex",
        "lesion_location"
    ],
    "context_completeness": {
        "total_fields": 3,
        "available_fields": 0,
        "missing_fields": 3,
        "score": 0.0,
        "status": "INCOMPLETE",
        "score_interpretation": "DATA_AVAILABILITY_ONLY"
    },
    "reliability_flags": [],
    "warnings": [
        "SEGMENTATION_NOT_VALIDATED_AGAINST_EXPERT_MASKS",
        "VISUAL_FEATURES_ARE_EXPERIMENTAL",
        "NO_CLINICAL_RISK_SCORE_GENERATED",
        "MANUAL_REVIEW_REQUIRED"
    ],
    "feature_status": "EXPERIMENTAL_UNVALIDATED",
    "clinical_constraints": {
        "independent_diagnosis": false,
        "automatic_acceptance": false,
        "requires_review": true


In [ ]:

# ============================================================
# CELL 6: EXPORT & DOWNSTREAM INTEGRATION TESTING
# ============================================================

import os
import json
import pandas as pd

# Output directories
OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "outputs", "reasoning_context"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR, "reports", "reasoning_context"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Output paths
JSON_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "reasoning_context_sample_logs.json"
)

CSV_OUTPUT = os.path.join(
    REPORT_DIR,
    "reasoning_context_sample_summary.csv"
)

# 1. Export complete context records
with open(JSON_OUTPUT, "w") as f:
    json.dump(
        context_records,
        f,
        indent=4,
        default=str
    )

# 2. Generate CSV summary
summary_rows = []

for record in context_records:

    summary_rows.append({
        "image_id": record["image_id"],
        "age": record["patient_context"]["age"],
        "sex": record["patient_context"]["sex"],
        "lesion_location": record["lesion_context"]["location"],
        "missing_information": ", ".join(
            record["missing_information"]
        ),
        "completeness_score": record[
            "context_completeness"
        ]["score"],
        "completeness_status": record[
            "context_completeness"
        ]["status"],
        "analysis_status": record[
            "lesion_context"
        ]["analysis_status"],
        "feature_status": record[
            "lesion_context"
        ]["feature_status"],
        "review_required": record[
            "clinical_constraints"
        ]["requires_review"],
        "automatic_acceptance": record[
            "clinical_constraints"
        ]["automatic_acceptance"]
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    CSV_OUTPUT,
    index=False
)

# 3. Reload exported files
with open(JSON_OUTPUT, "r") as f:
    exported_records = json.load(f)

exported_summary = pd.read_csv(CSV_OUTPUT)

# 4. Integration tests
tests = {}

tests["Six JSON records"] = len(exported_records) == 6

tests["Six CSV records"] = len(exported_summary) == 6

tests["Safety evidence present"] = all(
    "safety_agent" in r["downstream_evidence"]
    for r in exported_records
)

tests["Coordinator evidence present"] = all(
    "coordinator" in r["downstream_evidence"]
    for r in exported_records
)

tests["Missing information preserved"] = all(
    len(r["missing_information"]) == 3
    for r in exported_records
)

tests["Review status preserved"] = all(
    r["lesion_context"]["analysis_status"]
    == "REVIEW_REQUIRED"
    for r in exported_records
)

tests["Experimental status preserved"] = all(
    r["lesion_context"]["feature_status"]
    == "EXPERIMENTAL_UNVALIDATED"
    for r in exported_records
)

tests["Automatic acceptance disabled"] = all(
    r["clinical_constraints"]["automatic_acceptance"]
    is False
    for r in exported_records
)

tests["Manual review required"] = all(
    r["clinical_constraints"]["requires_review"]
    is True
    for r in exported_records
)

tests["JSON and CSV image IDs match"] = (
    set(r["image_id"] for r in exported_records)
    == set(exported_summary["image_id"])
)

# 5. Print verification report
print("=" * 65)
print("VELOURA PHASE 6 FINAL VERIFICATION")
print("=" * 65)

for test_name, passed in tests.items():
    print(
        f"{test_name}:",
        "PASS" if passed else "FAIL"
    )

print("\nTOTAL TESTS:", len(tests))
print("PASSED:", sum(tests.values()))
print("FAILED:", len(tests) - sum(tests.values()))

print("\nOUTPUT FILES:")
print("JSON:", JSON_OUTPUT)
print("CSV:", CSV_OUTPUT)

print("\nCSV SUMMARY:")
print(summary_df.to_string(index=False))

# Final assertion
assert all(tests.values()), "Phase 6 verification failed!"

print("\nSUCCESS: VELOURA PHASE 6 VERIFIED!")


VELOURA PHASE 6 FINAL VERIFICATION
Six JSON records: PASS
Six CSV records: PASS
Safety evidence present: PASS
Coordinator evidence present: PASS
Missing information preserved: PASS
Review status preserved: PASS
Experimental status preserved: PASS
Automatic acceptance disabled: PASS
Manual review required: PASS
JSON and CSV image IDs match: PASS

TOTAL TESTS: 10
PASSED: 10
FAILED: 0

OUTPUT FILES:
JSON: /content/drive/MyDrive/Veloura_Project/outputs/reasoning_context/reasoning_context_sample_logs.json
CSV: /content/drive/MyDrive/Veloura_Project/reports/reasoning_context/reasoning_context_sample_summary.csv

CSV SUMMARY:
    image_id  age  sex lesion_location       missing_information  completeness_score completeness_status analysis_status           feature_status  review_required  automatic_acceptance
ISIC_0033038 None None            None age, sex, lesion_location                 0.0          INCOMPLETE REVIEW_REQUIRED EXPERIMENTAL_UNVALIDATED             True                 False
ISI